hussain murtaza

1302250362


# Loan Approval: Traditional Programming vs Machine Learning
**Traditional:** we write the rules. **ML:** the computer learns the rules from past data.

## Step 1: Read both Excel files and sync them
Both files hold the same applicant data. If you edit a value (or add / delete an applicant) in EITHER file, run this notebook again: this step finds the edit and copies it into the other file.
- Inside each file, the `Results` sheet follows `New_Applicants` by itself (live Excel formulas).
- Between the two files, this notebook does the copying.

In [1]:
import os, shutil
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.comments import Comment
from openpyxl.utils import get_column_letter
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# In Colab: upload BOTH excel files first (left sidebar > Files)
IN_FILE = 'loan_applicants_input.xlsx'
OUT_FILE = 'loan_applicants_output.xlsx'

In [2]:
FIELDS = ['Age', 'Monthly_Income', 'Credit_Score', 'Loan_Amount', 'Existing_EMI']
HIST_FIELDS = FIELDS + ['Approved']
RESULT_COLS = ['Applicant_ID'] + FIELDS + ['Traditional_Decision', 'ML_Decision', 'Same_Decision']
RESULT_LETTER = {'Age': 'B', 'Monthly_Income': 'C', 'Credit_Score': 'D', 'Loan_Amount': 'E', 'Existing_EMI': 'F'}
STATE_SHEET = '_sync_state'   # hidden sheet that remembers the last synced values


def clean_value(v, field, where):
    """Turn one cell into a clean python value, or stop with a clear message."""
    if field == 'Approved':
        s = str(v).strip().capitalize()
        if s not in ('Yes', 'No'):
            raise ValueError(f'{where}: Approved must be Yes or No, found {v!r}')
        return s
    if isinstance(v, bool):
        raise ValueError(f'{where}: {field} must be a number, found {v!r}')
    if isinstance(v, str):
        try:
            v = float(v.replace(',', '').strip())
        except ValueError:
            raise ValueError(f'{where}: {field} must be a number, found {v!r}')
    if not isinstance(v, (int, float)) or v != v:
        raise ValueError(f'{where}: {field} must be a number, found {v!r}')
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return v


def read_table(path, sheet, fields):
    """Read one sheet. Returns (ids in sheet order, {id: {field: value}}) or None if the sheet is missing.
    A cell that holds a formula is a LINK, so it is stored as None (it is not an independent value)."""
    wb = load_workbook(path)
    if sheet not in wb.sheetnames:
        return None
    ws = wb[sheet]
    headers = {}
    for c in range(1, ws.max_column + 1):
        if ws.cell(1, c).value is not None:
            headers[ws.cell(1, c).value] = c
    for h in ['Applicant_ID'] + fields:
        if h not in headers:
            raise ValueError(f'{path} / {sheet}: column "{h}" not found in row 1')
    order, rows = [], {}
    for r in range(2, ws.max_row + 1):
        raw_id = ws.cell(r, headers['Applicant_ID']).value
        raw = {f: ws.cell(r, headers[f]).value for f in fields}
        if raw_id is None and all(v is None for v in raw.values()):
            continue                                   # completely empty row
        where = f'{path} / {sheet} / row {r}'
        if raw_id is None:
            raise ValueError(f'{where}: Applicant_ID is empty')
        aid = clean_value(raw_id, 'Applicant_ID', where)
        if not isinstance(aid, int):
            raise ValueError(f'{where}: Applicant_ID must be a whole number, found {raw_id!r}')
        if aid in rows:
            raise ValueError(f'{where}: Applicant_ID {aid} appears twice')
        row = {}
        for f in fields:
            v = raw[f]
            if isinstance(v, str) and v.startswith('='):
                row[f] = None                           # linked cell
            elif v is None:
                raise ValueError(f'{where}: {f} is empty')
            else:
                row[f] = clean_value(v, f, where)
        order.append(aid)
        rows[aid] = row
    return order, rows


def read_state(path):
    """Read the hidden sheet with the values from the last sync (None if there is none)."""
    wb = load_workbook(path)
    if STATE_SHEET not in wb.sheetnames:
        return None
    state = {'New': {}, 'History': {}}
    for r in wb[STATE_SHEET].iter_rows(min_row=2, values_only=True):
        if r[0] in state and r[1] is not None:
            vals = dict(zip(HIST_FIELDS, r[2:8]))
            keep = FIELDS if r[0] == 'New' else HIST_FIELDS
            state[r[0]][int(r[1])] = {f: vals[f] for f in keep}
    return state


def merge_table(table, copies, snap, fields, log):
    """Combine all copies of one table into ONE set of values.
    - a value that differs from the last synced value is an edit, and the edit wins
    - a row that is gone from a copy was deleted; a new row is added everywhere
    - if two copies were edited to different values, the most recently saved file wins"""
    ranked = sorted(copies, key=lambda c: (-c['mtime'], c['rank']))
    all_ids = set()
    for c in copies:
        all_ids |= set(c['rows'])
    if snap is not None:
        all_ids |= set(snap)
    merged = {}
    for aid in sorted(all_ids):
        holders = [c for c in ranked if aid in c['rows']]
        missing = [c['name'] for c in copies if aid not in c['rows']]
        base = None
        if snap is not None and aid in snap:
            if missing:
                log.append(f'{table} ID {aid}: row deleted in {", ".join(missing)} -> removed from all copies')
                continue
            base = snap[aid]
        elif snap is not None:
            log.append(f'{table} ID {aid}: new row found in {", ".join(c["name"] for c in holders)} -> added to all copies')
        elif missing:
            log.append(f'{table} ID {aid}: row missing in {", ".join(missing)} -> added to all copies')
        row = {}
        for f in fields:
            vals = [(c, c['rows'][aid][f]) for c in holders if c['rows'][aid][f] is not None]
            if base is not None:
                changed = [(c, v) for c, v in vals if v != base[f]]
                if not changed:
                    row[f] = base[f]
                    continue
                row[f] = changed[0][1]
                distinct = sorted({v for _, v in changed}, key=str)
                if len(distinct) > 1:
                    log.append(f'{table} ID {aid}: CONFLICT in {f} (copies disagree: {distinct}) -> kept {row[f]} from the most recently saved file')
                else:
                    src = ', '.join(c['name'] for c, v in changed)
                    log.append(f'{table} ID {aid}: {f} changed {base[f]} -> {row[f]} (edited in {src})')
            else:
                if not vals:
                    raise ValueError(f'{table} ID {aid}: no value found for {f}')
                row[f] = vals[0][1]
                distinct = sorted({v for _, v in vals}, key=str)
                if len(distinct) > 1:
                    log.append(f'{table} ID {aid}: copies disagree in {f} {distinct} (first sync, no earlier snapshot) -> kept {row[f]} from the most recently saved file')
        merged[aid] = row
    order, seen = [], set()
    for c in sorted(copies, key=lambda c: c['rank']):
        for aid in c['order']:
            if aid in merged and aid not in seen:
                order.append(aid)
                seen.add(aid)
    return merged, order


def update_sheet(ws, merged, order, fields):
    """Write the merged values into an existing sheet (keeps the sheet's own row order and any extra columns)."""
    headers = {ws.cell(1, c).value: c for c in range(1, ws.max_column + 1) if ws.cell(1, c).value is not None}
    id_col = headers['Applicant_ID']
    seen, to_delete = set(), []
    for r in range(2, ws.max_row + 1):
        v = ws.cell(r, id_col).value
        if v is None:
            continue
        aid = clean_value(v, 'Applicant_ID', f'{ws.title} row {r}')
        if aid in merged:
            for f in fields:
                cell = ws.cell(r, headers[f])
                if not (isinstance(cell.value, str) and cell.value.startswith('=')):   # never overwrite a link
                    cell.value = merged[aid][f]
            seen.add(aid)
        else:
            to_delete.append(r)
    for r in reversed(to_delete):
        ws.delete_rows(r)
    last = 1
    for r in range(2, ws.max_row + 1):
        if ws.cell(r, id_col).value is not None:
            last = r
    for aid in order:
        if aid not in seen:
            last += 1
            ws.cell(last, id_col).value = aid
            for f in fields:
                ws.cell(last, headers[f]).value = merged[aid][f]


def tree_formula(model, features, r, node=0):
    """Write the learned decision tree as an Excel IF formula for row r."""
    t = model.tree_
    if t.children_left[node] == -1:
        return '"%s"' % model.classes_[t.value[node][0].argmax()]
    col = RESULT_LETTER[features[t.feature[node]]]
    thr = repr(float(t.threshold[node]))
    left = tree_formula(model, features, r, t.children_left[node])
    right = tree_formula(model, features, r, t.children_right[node])
    return f'IF({col}{r}<={thr},{left},{right})'


def save_workbook(path, model, features):
    """Save the merged data + a live Results sheet + the hidden snapshot into one workbook file."""
    arial, bold = Font(name='Arial', size=10), Font(name='Arial', size=10, bold=True)
    wb = load_workbook(path)
    update_sheet(wb['New_Applicants'], merged_new, order_new, FIELDS)
    update_sheet(wb['History'], merged_hist, order_hist, HIST_FIELDS)

    na = wb['New_Applicants']
    na_letter = {na.cell(1, c).value: get_column_letter(c) for c in range(1, na.max_column + 1)}

    if 'Results' in wb.sheetnames:
        del wb['Results']
    ws = wb.create_sheet('Results', 2)
    ws.append(RESULT_COLS)
    for i, aid in enumerate(order_new, start=2):
        ws.cell(i, 1).value = aid
        for col, f in enumerate(FIELDS, start=2):      # B..F follow New_Applicants by Applicant_ID
            L = na_letter[f]
            ws.cell(i, col).value = f'=INDEX(New_Applicants!{L}:{L},MATCH($A{i},New_Applicants!${na_letter["Applicant_ID"]}:${na_letter["Applicant_ID"]},0))'
        ws.cell(i, 7).value = f'=IF(AND(D{i}>=650,(C{i}-F{i})>=30000,E{i}<=4*12*C{i}),"Yes","No")'
        ws.cell(i, 8).value = '=' + tree_formula(model, features, i)
        ws.cell(i, 9).value = f'=G{i}=H{i}'
    ws['G1'].comment = Comment('Rule: Credit_Score >= 650 AND (Monthly_Income - Existing_EMI) >= 30000 AND Loan_Amount <= 4*12*Monthly_Income. Recalculates by itself when B:F change.', 'Notebook')
    ws['H1'].comment = Comment('Decision tree learned from the History sheet, written as an Excel formula. Recalculates by itself when B:F change. Re-run the notebook to retrain after History changes.', 'Notebook')
    for c in range(1, 10):
        ws.column_dimensions[get_column_letter(c)].width = 20
    ws.freeze_panes = 'A2'

    if STATE_SHEET in wb.sheetnames:
        del wb[STATE_SHEET]
    st = wb.create_sheet(STATE_SHEET)
    st.append(['Table', 'Applicant_ID'] + HIST_FIELDS)
    for aid in order_new:
        st.append(['New', aid] + [merged_new[aid][f] for f in FIELDS] + [None])
    for aid in order_hist:
        st.append(['History', aid] + [merged_hist[aid][f] for f in HIST_FIELDS])
    st.sheet_state = 'hidden'

    for sheet in wb.worksheets:
        for row in sheet.iter_rows():
            for cell in row:
                if cell.value is not None:
                    cell.font = bold if cell.row == 1 else arial
    wb.active = 0
    wb.calculation.fullCalcOnLoad = True
    tmp = path + '.tmp'
    wb.save(tmp)
    os.replace(tmp, path)

In [3]:
log = []   # a list of everything the sync finds, printed at the end

if not os.path.exists(IN_FILE):
    raise FileNotFoundError('Upload loan_applicants_input.xlsx first')
out_missing = not os.path.exists(OUT_FILE)   # first run: the output file is created later, after the input passes all checks
if out_missing:
    log.append('Output file did not exist, so it will be created from the input file.')

# read every copy of the applicant data: both files, sheets New_Applicants and Results
new_copies, hist_copies = [], []
for path, label in [(IN_FILE, 'input file'), (OUT_FILE, 'output file')]:
    if not os.path.exists(path):
        continue
    mtime = os.path.getmtime(path)
    for sheet in ['New_Applicants', 'Results']:
        t = read_table(path, sheet, FIELDS)
        if t is None and sheet == 'New_Applicants':
            raise ValueError(f'{path}: sheet New_Applicants not found')
        if t is not None:
            new_copies.append({'name': f'{label} ({sheet})', 'mtime': mtime, 'rank': len(new_copies), 'order': t[0], 'rows': t[1]})
    t = read_table(path, 'History', HIST_FIELDS)
    if t is None:
        raise ValueError(f'{path}: sheet History not found')
    hist_copies.append({'name': f'{label} (History)', 'mtime': mtime, 'rank': len(hist_copies), 'order': t[0], 'rows': t[1]})

# values from the last sync (hidden sheet) - used to find out WHAT was edited
state_in = read_state(IN_FILE)
state_out = None if out_missing else read_state(OUT_FILE)
if state_in and state_out and state_in != state_out:
    log.append('WARNING: the two files remember different last-sync values (one may be an old copy).')
state = state_in or state_out
if state is None:
    log.append('First sync: no earlier snapshot found.')

merged_new, order_new = merge_table('New_Applicants', new_copies, state['New'] if state else None, FIELDS, log)
merged_hist, order_hist = merge_table('History', hist_copies, state['History'] if state else None, HIST_FIELDS, log)

print('SYNC REPORT')
for line in (log if log else ['No changes found - both files already match.']):
    print(' -', line)

# the final, merged data as tables
new = pd.DataFrame([{'Applicant_ID': a, **merged_new[a]} for a in order_new])
hist = pd.DataFrame([{'Applicant_ID': a, **merged_hist[a]} for a in order_hist])
if len(new) == 0 or len(hist) < 10:
    raise ValueError('Need at least 1 new applicant and 10 history rows')
print(len(new), 'new applicants,', len(hist), 'history rows')
new.head()

SYNC REPORT
 - No changes found - both files already match.
20 new applicants, 200 history rows


,Applicant_ID,Age,Monthly_Income,Credit_Score,Loan_Amount,Existing_EMI
0,1001,54,48489,592,653320,14016
1,1002,39,115311,680,1099863,25236
2,1003,58,96852,776,463562,15892
3,1004,59,68836,609,103985,7810
4,1005,43,113138,516,795983,6870


## Step 2: Traditional programming (rules written by us)

In [4]:
def traditional_rule(r):
    if r.Credit_Score >= 650 and (r.Monthly_Income - r.Existing_EMI) >= 30000 and r.Loan_Amount <= 4 * 12 * r.Monthly_Income:
        return 'Yes'
    return 'No'

new['Traditional_Decision'] = new.apply(traditional_rule, axis=1)
new[['Applicant_ID','Traditional_Decision']].head()

,Applicant_ID,Traditional_Decision
0,1001,No
1,1002,Yes
2,1003,Yes
3,1004,No
4,1005,No


## Step 3: Machine Learning (computer learns from History sheet)

In [5]:
features = ['Age','Monthly_Income','Credit_Score','Loan_Amount','Existing_EMI']
X_train, X_test, y_train, y_test = train_test_split(hist[features], hist['Approved'], test_size=0.25, random_state=42)

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
print('ML accuracy on unseen history data:', round(accuracy_score(y_test, model.predict(X_test)),2))

new['ML_Decision'] = model.predict(new[features])

ML accuracy on unseen history data: 0.66


## Step 4: Compare and write to both Excel files

In [6]:
new['Same_Decision'] = new['Traditional_Decision'] == new['ML_Decision']
print('Both methods agree on', new['Same_Decision'].sum(), 'of', len(new), 'applicants')

# write BOTH files: same data in both, Results sheet with live formulas, hidden snapshot for the next sync
if not os.path.exists(OUT_FILE):
    shutil.copy(IN_FILE, OUT_FILE)             # create the output file only now, after all checks passed
for path in [IN_FILE, OUT_FILE]:
    save_workbook(path, model, features)
print('Saved', IN_FILE, 'and', OUT_FILE)
new

Both methods agree on 15 of 20 applicants


Saved loan_applicants_input.xlsx and loan_applicants_output.xlsx


,Applicant_ID,Age,Monthly_Income,Credit_Score,Loan_Amount,Existing_EMI,Traditional_Decision,ML_Decision,Same_Decision
0,1001,54,48489,592,653320,14016,No,No,True
1,1002,39,115311,680,1099863,25236,Yes,Yes,True
2,1003,58,96852,776,463562,15892,Yes,Yes,True
3,1004,59,68836,609,103985,7810,No,Yes,False
4,1005,43,113138,516,795983,6870,No,Yes,False
5,1006,47,60462,517,368968,9674,No,No,True
6,1007,30,29665,798,1095469,17923,No,Yes,False
7,1008,52,18763,539,2170634,7274,No,No,True
8,1009,47,113492,722,490166,27425,Yes,Yes,True
9,1010,56,115776,634,102959,14395,No,Yes,False


## Conclusion
- Traditional: rules are fixed; we must know them beforehand and update by hand.
- ML: learns patterns from past data, adapts when data changes, but needs good data.
- Where they disagree, a human should review the case.